# Proyecto Fashion Peaks: 01 - Generador Unificado de Corpus
Este notebook genera el conjunto sintético de espectros gamma para el entrenamiento de redes neuronales convolucionales (CNN).
Permite generar cualquiera de las tres variantes:
- **v1**: Gráfico de línea estándar (clasificación en 40 intervalos de 500 unidades).
- **v2**: Gráfico con área bajo la curva sombreada en negro (`plt.fill_between`), para resaltar la región del pico.
- **v3**: Gráfico con área sombreada y etiquetas continuas de área (0 a 20.000) para **Regresión**.


In [ ]:
# 1. Configuración del entorno (compatible con Google Colab y Local)
import sys, os

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive')
    
    # MODIFICA esta ruta según donde ubiques la carpeta en tu Google Drive:
    PATH_DRIVE = '/content/gdrive/MyDrive/Fashion_Peaks_Colab'
    if os.path.exists(PATH_DRIVE):
        os.chdir(PATH_DRIVE)
    print(f"-> Conectado a Google Colab. Directorio de trabajo: {os.getcwd()}")
else:
    print(f"-> Entorno local detectado. Directorio de trabajo: {os.getcwd()}")


In [ ]:
# 2. Importación de librerías
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm


In [ ]:
# 3. Parámetros de simulación y calibración energética
# Calibración del canal a energía (keV): E = aa * canal + bb
nroBin = 1900
aa = 0.355574
bb = -4.26897
channel = np.linspace(0, nroBin, num=nroBin)
energy = channel * aa + (bb + 0.8)

# Energía del fotopico de Cesio-137
E_Cs137 = 661.657

# ==============================================================================
# CONFIGURACIÓN DEL CORPUS A GENERAR:
# 'v1_linea'     : Trazo negro simple, etiquetas discretas (clasificación)
# 'v2_sombreado' : Trazo negro + relleno negro bajo la curva, etiquetas discretas
# 'v3_regresion' : Trazo negro + relleno negro, etiqueta continua de área (0 - 20000)
# ==============================================================================
MODO = 'v2_sombreado'  # <--- Cambiá aquí entre 'v1_linea', 'v2_sombreado', 'v3_regresion'
N_TRAIN = 2000         # Cantidad de espectros a sintetizar


In [ ]:
# 4. Carga de datos base (Fondo natural y Cesio puro)
file_base = 'fileBase_2000x1900.dat'
file_cesio = 'fileCesio_2000x1900.dat'

# Si los archivos están en la carpeta 'datos/', los buscamos allí
if not os.path.exists(file_base) and os.path.exists(os.path.join('datos', file_base)):
    file_base = os.path.join('datos', file_base)
    file_cesio = os.path.join('datos', file_cesio)

cant_spectra = 2000
num_bins = 1900

print(f"Cargando {file_base}...")
lee_base = np.loadtxt(file_base)
base_spectra = lee_base.reshape((cant_spectra, num_bins))

print(f"Cargando {file_cesio}...")
lee_cesio = np.loadtxt(file_cesio)
cesio_spectra = lee_cesio.reshape((cant_spectra, num_bins))

print(f"Carga exitosa: Base {base_spectra.shape}, Cesio {cesio_spectra.shape}")


In [ ]:
# 5. Visualización previa de los datos de entrada
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(energy, base_spectra[0], 'k', label='Fondo natural')
plt.xlim(0, 700)
plt.xlabel("Energía [keV]")
plt.ylabel("Cuentas")
plt.title("Espectro de Fondo")
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(energy, cesio_spectra[0], 'r', label='Cs-137 puro')
plt.axvline(x=E_Cs137, color='g', linestyle='--', label=f'Cs-137 ({E_Cs137} keV)')
plt.xlim(600, 700)
plt.xlabel("Energía [keV]")
plt.ylabel("Cuentas")
plt.title("Pico de Cesio-137")
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# 6. Síntesis aleatoria Monte Carlo (Fondo + Cesio)
np.random.seed(42)

x_train = np.zeros((N_TRAIN, num_bins))
y_train = np.zeros((N_TRAIN, num_bins))

print(f"Generando {N_TRAIN} mezclas sintéticas de espectros...")
for jj in tqdm(range(N_TRAIN)):
    idx = np.random.randint(0, cant_spectra)
    idy = np.random.randint(0, cant_spectra)

    base = base_spectra[idx]
    cesio = cesio_spectra[idy] * 0.3  # Escala física

    spectrum = base + cesio
    factor = spectrum.max() * 1.001

    x_train[jj] = spectrum / factor
    y_train[jj] = cesio / factor

print("Mezclas generadas correctamente.")


In [ ]:
# 7. Generación de imágenes y archivo labels.csv
def punto_medio_intervalo(n: int) -> int:
    """Retorna el punto medio del intervalo de 500 unidades para clasificación (0 a 20.000)"""
    if n <= 0:
        return 250
    k = min(39, (n - 1) // 500)
    return k * 500 + 250

if MODO == 'v3_regresion':
    output_dir = "imagenes_entrenamiento_regresion"
else:
    output_dir = "imagenes_entrenamiento"

os.makedirs(output_dir, exist_ok=True)

width_px, height_px = 32, 64
dpi = 100
figsize = (width_px / dpi, height_px / dpi)

registros = []
print(f"Generando imágenes en modo '{MODO}' dentro de '{output_dir}'...")

for idx in tqdm(range(N_TRAIN)):
    area_continua = int(y_train[idx].sum() * 1000)
    
    if MODO == 'v3_regresion':
        label = area_continua  # Valor continuo para regresión
    else:
        label = punto_medio_intervalo(area_continua)  # Clase binned (250, 750, ...)
    
    fig = plt.figure(figsize=figsize, dpi=dpi)
    plt.plot(energy, x_train[idx], 'k')
    
    # Relleno del área en v2 y v3
    if MODO in ['v2_sombreado', 'v3_regresion']:
        plt.fill_between(energy, x_train[idx], color='black')
        
    plt.xlim(659, 664)  # Ventana de interés en torno a 661.66 keV
    plt.ylim(0, 1.2)
    plt.xticks([])
    plt.yticks([])
    plt.axis('off')
    
    filename = f"spectro_{idx:05d}.png"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, dpi=dpi, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    
    registros.append((filename, label))

df = pd.DataFrame(registros, columns=["filename", "label"])
csv_path = os.path.join(output_dir, "labels.csv")
df.to_csv(csv_path, index=False)

print(f"¡Listo! Se guardaron {N_TRAIN} imágenes y {csv_path}")
